In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt, Production
from pydantic import BaseModel

In [2]:
from agents.utils import get_connection

conn = get_connection()
cur = conn.cursor()

for table in ["Usage", "Chat"]:
    cur.execute("""
        SELECT TABLE_NAME
        FROM INFORMATION_SCHEMA.Tables
        WHERE TABLE_TYPE='BASE TABLE'
          AND TABLE_SCHEMA='SQLUser'
          AND TABLE_NAME=?
    """, (table,))
    if cur.fetchone():
        cur.execute(f"DROP TABLE IF EXISTS SQLUser.{table}")

conn.commit()

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [3]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')


Load started on 04/09/2026 11:02:11
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/09/2026 11:02:11
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/09/2026 11:02:12
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/09/2026 11:02:12
Loading file Agents.Operation.ToolkitUtilities.cls as udl
Compiling class Agents.Operation.ToolkitUtilities
Compiling routine Agents.Operation.ToolkitUtilities.1
Load finished successfully.

Load started on 04/09/2026 11:02:12
Loading file Agents.Message.ToolRequest.cls as ud

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [4]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [5]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [6]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [7]:
bond_system = Prompt(name = 'Agent007', text = 'You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [8]:
bond_system = Prompt(name = 'Agent007', text = 'Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [9]:
Prompt('Agent007') == bond_system

True

In [10]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [11]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [12]:
Prompt('Agent007').delete()
try:
    prompt = Prompt("Agent007")
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [13]:
molly = Agent(name='Molly', model='gpt-5')
Production('AgentSpace', [molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 04/09/2026 11:02:14
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/09/2026 11:02:14
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/09/2026 11:02:14
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/09/2026 11:02:15
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/09/2026 11:02:15
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are great summer-friendly hiking spots within about an hour of Boston:\n\n- Blue Hills Reservation (Milton/Canton): Skyline Trail (up to 9–10 miles, strenuous) or shorter loops to Great Blue Hill tower; rocky ridges, city views.\n- Middlesex Fells Reservation (Medford/Winchester/Stoneham): Skyline (6.9 miles, moderate) or Rock Circuit (4 miles, rugged); ponds, lookouts. Parking fills early.\n- Walden Pond & Town Forest (Concord): Easy 1.7-mile pond loop plus woodland trails; swimming options.\n- Minute Man National Historical Park (Lexington/Concord): Battle Road Trail (5 miles, easy); shaded, historic.\n- Noanet Woodlands (Dover): 17+ miles of easy–moderate forest paths; Noanet Peak views of Boston skyline.\n- Rocky Woods (Medfield): 6+ miles, easy–moderate; ponds and shaded loops.\n- World’s End (Hingham): 3–4 miles, easy; ocean breezes and harbor views. Reservation system on busy days; entry fee.\n- Lynn Woods Reservation (Lynn): 30+ miles; Dungeon Rock, Stone Tower; moderate,

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [14]:
Agent('Molly') == molly

True

In [15]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             reasoning_effort='low',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 04/09/2026 11:02:46
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 04/09/2026 11:02:47
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/09/2026 11:02:47
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/09/2026 11:02:47
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/09/2026 11:02:47
Loading file Agents.Ope

In [16]:
Production('AgentSpace', [molly, alex]).start()


Load started on 04/09/2026 11:02:50
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/09/2026 11:02:50
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/09/2026 11:02:50
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/09/2026 11:02:50
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/09/2026 11:02:50
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

In [17]:
molly(message='Which tables do we have in IRIS in the Agents namespace?')

'{"text": "The Agents namespace contains these tables: SQLUser.Agent, SQLUser.AgentToolkit, SQLUser.Chat, SQLUser.Prompt, SQLUser.TestModel, SQLUser.Toolkit, SQLUser.ToolUsage, SQLUser.Usage.", "reasoning": "Used the provided IRIS list_tables result for the Agents namespace."}'

In [18]:
molly(message='What is the weather today?', chat=context)

'{"text": "Washington DC: Cloudy. High 26\\u00b0, Low 13\\u00b0.", "reasoning": "Use the provided tool result for Utilities.weather showing Washington DC conditions."}'

In [19]:
molly(message='Recommend some good food spots for lunch', chat='travel', reasoning_effort='high')

'{"text": "Here are solid lunch spots around DC:\\n- Near the Mall/Penn Quarter: Old Ebbitt Grill (classic American, oysters); Zaytinya (Mediterranean mezze); Daikaya (ramen).\\n- Fast-casual/local: Shouk (plant-based Israeli); Falafel Inc (cheap, tasty falafel); CAVA (Mediterranean bowls); Wiseguy Pizza (big slices).\\n- Seafood: The Salt Line (Navy Yard; lobster roll, oysters); Hank\'s Oyster Bar (Dupont or The Wharf).\\n- Iconic/only-in-DC: Ben\'s Chili Bowl (U St half-smokes); Founding Farmers (Foggy Bottom; American).\\n- BBQ/Ethiopian: Federalist Pig (Adams Morgan BBQ); Chercher (Shaw Ethiopian).\\n- Pizza/sit-down: All-Purpose (Shaw or Navy Yard; crispy pies).\\nTell me your neighborhood, cuisine and budget, and whether you want quick counter-service or a sit-down meal, and I\\u2019ll narrow it to the best 2\\u20133 nearby.", "reasoning": "User is in Washington DC; no tool is needed. Provide a concise, diverse set of lunch options by cuisine and neighborhood, then ask for prefer

In [20]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 04/09/2026 11:04:07
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 04/09/2026 11:04:08
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "L\'Ardente", "cuisine": "Italian"}, {"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "Sfoglina", "cuisine": "Italian"}, {"name": "The Red Hen", "cuisine": "Italian"}, {"name": "Centrolina", "cuisine": "Italian"}, {"name": "Filomena Ristorante", "cuisine": "Italian"}, {"name": "Daikaya", "cuisine": "Japanese"}, {"name": "Sushi Taro", "cuisine": "Japanese"}, {"name": "Anju", "cuisine": "Korean"}, {"name": "Thip Khao", "cuisine": "Lao"}, {"name": "Maketto", "cuisine": "Cambodian/Taiwanese"}, {"name": "Tiger Fork", "cuisine": "Chinese"}], "reasoning": "Since you like Italian and Asian cuisines in Washington DC, here are well-regarded spots across both styles."}'

In [21]:
Chat('travel').usage()

'{"input_tokens": 2928, "output_tokens": 4595, "output_reasoning_tokens": 3840, "total_tokens": 7523}'

In [22]:
Production('AgentSpace').usage()

{'input_tokens': 4577,
 'output_tokens': 6737,
 'output_reasoning_tokens': 5312,
 'total_tokens': 11314}

In [23]:
molly.usage()

{'input_tokens': 4577,
 'output_tokens': 6737,
 'output_reasoning_tokens': 5312,
 'total_tokens': 11314}

In [24]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 4577,
 'output_tokens': 6737,
 'output_reasoning_tokens': 5312,
 'total_tokens': 11314}

In [25]:
Production('AgentSpace').delete()

Deleted production: User.AgentSpace

Deleting class Agents.REST.Dispatch.AgentSpaceCleaned up production-owned artifacts for: AgentSpace


In [26]:
Agent('Molly').delete()
try:
    molly('Hello')
except KeyError as e:
    print(e)


Deleting class Agents.Gateway.MollyService
Deleting class Agents.Process.Molly"No Agent found for 'Molly'"
